In [1]:



import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import torch
import torch.nn as nn
import gymnasium as gym

In [2]:

class DQN(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.input_size = input_size
        self.output_size = output_size
        self.net = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.SiLU(),
            nn.Linear(64,64),
            nn.SiLU(True),
            nn.Linear(64, output_size)
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class MemBuffer():
    def __init__(self, capacity=10000):
        self.state = []
        self.actions = []
        self.terminated = []
        self.rewards = []
        self.next_state = []
        self.capacity = capacity
        self.len = 0
    def push(self, s, a, t, r, s_next):
        if self.len >= self.capacity:
            self.state.pop(0)
            self.actions.pop(0)
            self.terminated.pop(0)
            self.rewards.pop(0)
            self.next_state.pop(0)
            self.len -= 1
        self.state.append(s)
        self.actions.append(a)
        self.terminated.append(t)
        self.rewards.append(r)
        self.next_state.append(s_next)
        self.len += 1
    def __len__(self):
        return self.len
    def get(self):
        return self.state, self.actions, self.terminated, self.rewards, self.next_state
    def sample(self, batch_size):
        idx = random.sample(range(self.len), batch_size)
        s = torch.stack([self.state[i] for i in idx])
        a = torch.tensor([self.actions[i] for i in idx], dtype=torch.long)
        t = torch.tensor([self.terminated[i] for i in idx], dtype=torch.float32)
        r = torch.tensor([self.rewards[i] for i in idx], dtype=torch.float32)
        s_next = torch.stack([self.next_state[i] for i in idx])
        return s, a, t, r, s_next
    def clear(self):
        self.state = []
        self.actions = []
        self.terminated = []
        self.rewards = []
        self.next_state = []
        self.len = 0


def select_action(model: nn.Module, s: torch.Tensor, noise_eps: float):
    with torch.no_grad():
        q = model(s.unsqueeze(0))
        q = (q - q.mean(dim=-1, keepdim=True)) / (q.std(dim=-1, keepdim=True) + 1e-6)
        q = q + noise_eps * torch.randn_like(q)
        probs = torch.softmax(q, dim=-1)
    return torch.multinomial(probs, 1).item()
# change it
def state_to_tensor(s):
    me, d, ace = s
    # print(me)
    me_t = torch.nn.functional.one_hot(torch.tensor(me).long(), num_classes=40)
    d_t = torch.nn.functional.one_hot(torch.tensor(d).long(), num_classes=11)
    ace = torch.nn.functional.one_hot(torch.tensor(ace).long(), num_classes=2)
    s_t = torch.cat([me_t, d_t, ace]).float()
    return s_t
def learn(buffer: MemBuffer, model: nn.Module, target_model: nn.Module,
          optim: torch.optim, gamma: float = 0.95, batch_size: int = 64):
    if len(buffer) < batch_size:
        return
    s, a, t, r, s_next = buffer.sample(batch_size)
    with torch.no_grad():
        q_next = target_model(s_next).max(dim=1).values
        y = r + gamma * (1.0 - t) * q_next
    q = model(s).gather(1, a.unsqueeze(1)).squeeze(1)
    loss = torch.nn.functional.mse_loss(q, y)
    optim.zero_grad()
    loss.backward()
    optim.step()


def rollout(env, model, input_size, output_size, episodes=500, target_sync=10):

    buffer = MemBuffer()
    # model = DQN(input_size, output_size)
    target_model = DQN(input_size, output_size)
    target_model.load_state_dict(model.state_dict())
    target_model.eval()
    optim = torch.optim.Adam(model.parameters(), lr=1e-3)

    ##
    K = 40
    trailing_Kwr_wins = []

    ##
    for episode in range(episodes):
        s, _ = env.reset()
        s = torch.tensor(s, dtype=torch.float32).flatten()
        done = False

        while not done:
            noise_eps = max(0.05, 0.2 - 0.15 * (episode / episodes))
            s_t = state_to_tensor(s)
            a = select_action(model, s_t, noise_eps)
            s_next, r, term, trunc, _ = env.step(a)
            done = bool(term or trunc)
            s_next = torch.tensor(s_next, dtype=torch.float32).flatten()
            buffer.push(state_to_tensor(s), a, float(term), float(r), state_to_tensor(s_next))
            s = s_next
            learn(buffer, model, target_model, optim)

        if episode % target_sync == 0:
            target_model.load_state_dict(model.state_dict())
        trailing_Kwr_wins.append(r)
        l_trailing = len(trailing_Kwr_wins[-K:])
        #print(f" {l_trailing} {sum(trailing_Kwr_wins[-K:])/ min(K,l_trailing):.4f} wr: {(torch.sum((torch.tensor(trailing_Kwr_wins[-K:]) > 0).float()) / min(K,l_trailing)):.3f}")
    return model
def eval_argmax(model, env, n=10000):
    wins = 0
    for _ in range(n):
        s, _ = env.reset()
        done = False
        while not done:
            with torch.no_grad():
                q = model(state_to_tensor(s).unsqueeze(0))
            a = int(q.argmax(dim=1).item())
            s, r, term, trunc, _ = env.step(a)
            done = term or trunc
        wins += (r > 0)
    return wins / n


env = gym.make('Blackjack-v1')
model = DQN(53, 2)
#before
print("argmax win rate:", eval_argmax(model, env))
model = rollout(env, model, 53, 2, 5000)
# after training:
print("argmax win rate:", eval_argmax(model, env))

argmax win rate: 0.3827


/tmp/ipykernel_739037/1425621685.py:72: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  me_t = torch.nn.functional.one_hot(torch.tensor(me).long(), num_classes=40)
/tmp/ipykernel_739037/1425621685.py:73: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  d_t = torch.nn.functional.one_hot(torch.tensor(d).long(), num_classes=11)
/tmp/ipykernel_739037/1425621685.py:74: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ace = torch.nn.functional.one_hot(torch.tensor(ace).long(), num_classes=2)


argmax win rate: 0.426


In [3]:
# a = [-1,2,3,4,5,7,8]
# k = 4
# z = a[-k:]
# l = (torch.tensor(a) > 0).float()
# l

In [4]:
%%timeit
y = model(torch.randn(53))

66.9 μs ± 5.67 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [5]:
# s, _ = e.reset()
# print(s)
    


In [6]:
# me, dealer, usable_ace = s
# me, dealer, usable_ace